# CFR+ 69-Claim A100 Streaming Profile

Run this on the A100 before starting the overnight 69-claim CFR+ job. It profiles the streamed GPU traversal path with `commit_records=False`, so it does not fill replay buffers or write checkpoints. The goal is to choose safe values for:

- root traversal batch size: how many root traversals are processed in one GPU call;
- streamed live row budget: the maximum live frontier rows before recursive splitting;
- traverser action chunk size: how many expanded traverser action edges are evaluated per child chunk;
- action sampling schedule: e.g. `16`, `32`, `69,32`, `69,69,16`.

Important: if the eventual training run uses `traversals_per_player=4096`, then a `traversal_batch_size` above 4096 will not help that run, because the trainer has only 4096 root traversals to process per traverser per iteration. Larger root batches only matter if `traversals_per_player` is also larger.

In [ ]:
from __future__ import annotations

import gc
import json
import math
import os
import sys
import time
from pathlib import Path

import numpy as np
import pandas as pd
import torch

REPO_ROOT = Path.cwd()
while not (REPO_ROOT / 'liars_poker').exists() and REPO_ROOT != REPO_ROOT.parent:
    REPO_ROOT = REPO_ROOT.parent
if str(REPO_ROOT) not in sys.path:
    sys.path.insert(0, str(REPO_ROOT))

from liars_poker.core import GameSpec
from liars_poker.env import rules_for_spec
from liars_poker.algo.deep_cfr_plus import DeepCFRPlusTrainer

assert torch.cuda.is_available(), 'This profile notebook is intended for the A100 CUDA machine.'
device = torch.device('cuda')
torch.set_float32_matmul_precision('high')

free_bytes, total_bytes = torch.cuda.mem_get_info(device)
print('python:', sys.executable)
print('torch:', torch.__version__)
print('cuda build:', torch.version.cuda)
print('gpu:', torch.cuda.get_device_name(device))
print('free / total GPU GiB:', round(free_bytes / 2**30, 2), '/', round(total_bytes / 2**30, 2))
print('repo:', REPO_ROOT)

## Shared Spec And Helpers

In [ ]:
SPEC = GameSpec(
    ranks=6,
    suits=4,
    hand_size=4,
    claim_kinds=('RankHigh', 'Pair', 'TwoPair', 'Trips', 'FullHouse', 'Quads'),
    suit_symmetry=True,
)

# These are intentionally small because profile traversals use commit_records=False.
# The models should match the intended overnight run, but replay capacity is irrelevant here.
BASE_TRAINER_KWARGS = dict(
    regret_hidden_sizes=(2048, 2048),
    strategy_hidden_sizes=(512, 512),
    device=device,
    seed=7,
    regret_buffer_capacity=4096,
    strategy_buffer_capacity=4096,
    learning_rate=1e-3,
    batch_size=4096,
    regret_train_steps=24,
    strategy_train_steps=6,
    strategy_weighting='linear',
    regret_positive_weight=0.5,
    validation_fraction=0.0,
    validation_buffer_capacity=1024,
    traversal_backend='gpu_native',
    device_replay=True,
    fused_optimizer=True,
    amp_dtype=None,
    compile_models=False,
)

def make_trainer(
    *,
    traversal_batch_size: int,
    schedule: tuple[int, ...] | None = None,
    sample_count: int | None = None,
    sample_fraction: float | None = None,
    baseline: str = 'call',
    sample_mode: str = 'hash',
    live_row_budget: int | None = None,
    action_chunk_size: int | None = None,
    seed: int = 7,
) -> DeepCFRPlusTrainer:
    kwargs = dict(BASE_TRAINER_KWARGS)
    kwargs.update(
        seed=seed,
        traversal_batch_size=traversal_batch_size,
        traverser_action_sample_schedule=schedule,
        traverser_action_sample_count=sample_count,
        traverser_action_sample_fraction=sample_fraction,
        traverser_action_baseline=baseline,
        traverser_action_sample_mode=sample_mode,
        traversal_streaming=True,
        traversal_live_row_budget=live_row_budget,
        traverser_action_chunk_size=action_chunk_size,
        traversal_record_flush_size=262_144,
    )
    trainer = DeepCFRPlusTrainer(SPEC, **kwargs)
    # CFR+ target scaling uses iteration; set it to a nonzero value for profile-only calls.
    trainer.iteration = 1
    return trainer

def ess_fraction(stats: dict) -> float:
    n = int(stats.get('regret_weight_count', 0) or 0)
    s1 = float(stats.get('regret_weight_sum', 0.0) or 0.0)
    s2 = float(stats.get('regret_weight_square_sum', 0.0) or 0.0)
    if n <= 0 or s2 <= 0.0:
        return float('nan')
    return (s1 * s1) / (n * s2)

def gib(value: int | float) -> float:
    return float(value) / 2**30

def profile_once(config: dict, *, traverser: int, warmup: bool = False) -> dict:
    trainer = make_trainer(**config)
    root_batch = int(config['traversal_batch_size'])
    if warmup:
        trainer._gpu_traverser.run_traversals(
            traverser,
            min(root_batch, 128),
            profile=False,
            commit_records=False,
        )
        torch.cuda.synchronize(device)
    gc.collect()
    torch.cuda.empty_cache()
    torch.cuda.reset_peak_memory_stats(device)
    torch.manual_seed(int(config.get('seed', 7)) + 10_000 * traverser)
    start = time.perf_counter()
    stats = trainer._gpu_traverser.run_traversals(
        traverser,
        root_batch,
        profile=True,
        commit_records=False,
    )
    torch.cuda.synchronize(device)
    wall_s = time.perf_counter() - start
    full_edges = int(stats.get('full_claim_edges', 0) or 0)
    sampled_edges = int(stats.get('sampled_claim_edges', 0) or 0)
    row = {
        **{k: v for k, v in config.items() if k != 'seed'},
        'traverser': traverser,
        'wall_s': wall_s,
        'root_traversals': root_batch,
        'regret_records': int(stats.get('regret_records', 0) or 0),
        'strategy_records': int(stats.get('strategy_records', 0) or 0),
        'full_claim_edges': full_edges,
        'sampled_claim_edges': sampled_edges,
        'claim_edge_fraction': sampled_edges / full_edges if full_edges else float('nan'),
        'regret_weight_ESS_fraction': ess_fraction(stats),
        'largest_regret_weight': float(stats.get('max_regret_weight', 0.0) or 0.0),
        'streamed_edge_chunks': int(stats.get('streamed_edge_chunks', 0) or 0),
        'streamed_row_splits': int(stats.get('streamed_row_splits', 0) or 0),
        'peak_rows': int(stats.get('peak_rows', 0) or 0),
        'peak_allocated_GiB': gib(stats.get('peak_allocated_bytes', 0) or 0),
        'peak_reserved_GiB': gib(stats.get('peak_reserved_bytes', 0) or 0),
        'records_per_s': (
            (int(stats.get('regret_records', 0) or 0) + int(stats.get('strategy_records', 0) or 0))
            / wall_s
            if wall_s > 0.0 else float('nan')
        ),
        'stats': stats,
    }
    del trainer
    gc.collect()
    torch.cuda.empty_cache()
    return row

def summarize_profiles(rows: list[dict]) -> pd.DataFrame:
    frame = pd.DataFrame(rows)
    cols = [
        'label', 'traverser', 'traversal_batch_size', 'schedule', 'sample_count',
        'sample_fraction', 'sample_mode', 'baseline', 'live_row_budget',
        'action_chunk_size', 'wall_s', 'root_traversals', 'regret_records',
        'strategy_records', 'claim_edge_fraction', 'regret_weight_ESS_fraction',
        'largest_regret_weight', 'streamed_edge_chunks', 'streamed_row_splits',
        'peak_rows', 'peak_allocated_GiB', 'peak_reserved_GiB', 'records_per_s',
    ]
    existing = [col for col in cols if col in frame.columns]
    return frame[existing].sort_values(['label', 'traverser']).reset_index(drop=True)

RULES = rules_for_spec(SPEC)
print('actions including CALL:', len(RULES.claims) + 1)
print('claim count:', len(RULES.claims))

## Candidate Overnight Traversal Profiles

These are the main candidates. The first pass uses moderate root batches so failures are cheap. If these are safe, run the larger row-budget sweep below.

In [ ]:
CANDIDATE_CONFIGS = [
    dict(
        label='cap16 hash, root 4096',
        traversal_batch_size=4096,
        schedule=(16,),
        sample_count=None,
        sample_fraction=None,
        baseline='call',
        sample_mode='hash',
        live_row_budget=65_536,
        action_chunk_size=8_192,
    ),
    dict(
        label='cap32 hash, root 4096',
        traversal_batch_size=4096,
        schedule=(32,),
        sample_count=None,
        sample_fraction=None,
        baseline='call',
        sample_mode='hash',
        live_row_budget=65_536,
        action_chunk_size=8_192,
    ),
    dict(
        label='cap48 hash, root 4096',
        traversal_batch_size=4096,
        schedule=(48,),
        sample_count=None,
        sample_fraction=None,
        baseline='call',
        sample_mode='hash',
        live_row_budget=65_536,
        action_chunk_size=8_192,
    ),
    dict(
        label='full-first then cap32 hash, root 4096',
        traversal_batch_size=4096,
        schedule=(69, 32),
        sample_count=None,
        sample_fraction=None,
        baseline='call',
        sample_mode='hash',
        live_row_budget=65_536,
        action_chunk_size=8_192,
    ),
    dict(
        label='full-first-two then cap16 hash, root 1024',
        traversal_batch_size=1024,
        schedule=(69, 69, 16),
        sample_count=None,
        sample_fraction=None,
        baseline='call',
        sample_mode='hash',
        live_row_budget=65_536,
        action_chunk_size=8_192,
    ),
]

candidate_rows = []
for config in CANDIDATE_CONFIGS:
    print('\n---', config['label'], '---')
    for traverser in (0, 1):
        row = profile_once(config, traverser=traverser, warmup=(len(candidate_rows) == 0))
        candidate_rows.append(row)
        print(
            f"p{traverser + 1}: wall={row['wall_s']:.3f}s "
            f"peak_reserved={row['peak_reserved_GiB']:.2f}GiB "
            f"edges={row['claim_edge_fraction']:.3f} "
            f"ESS={row['regret_weight_ESS_fraction']:.3f} "
            f"chunks={row['streamed_edge_chunks']} splits={row['streamed_row_splits']}"
        )

candidate_df = summarize_profiles(candidate_rows)
candidate_df.style.format(precision=4)

## Live Row Budget And Chunk Size Sweep

This is the cell that answers whether larger streamed budgets such as 131072 or 262144 are safe. It profiles the same action schedule at multiple row/chunk budgets. If the 262144 row-budget rows are fast and use comfortably below the A100 memory limit, use them for the overnight run. If row splits disappear but wall time worsens, prefer the smaller row budget.

In [ ]:
# Edit this one schedule to match the most promising candidate from the previous cell.
SWEEP_BASE = dict(
    label='full-first then cap32 hash',
    traversal_batch_size=4096,
    schedule=(69, 32),
    sample_count=None,
    sample_fraction=None,
    baseline='call',
    sample_mode='hash',
)

ROW_BUDGETS = [32_768, 65_536, 131_072, 262_144]
ACTION_CHUNKS = [8_192, 16_384, 32_768]

sweep_rows = []
for live_budget in ROW_BUDGETS:
    for chunk_size in ACTION_CHUNKS:
        config = dict(
            SWEEP_BASE,
            label=f"{SWEEP_BASE['label']}; rows={live_budget}; chunk={chunk_size}",
            live_row_budget=live_budget,
            action_chunk_size=chunk_size,
        )
        print('\n---', config['label'], '---')
        # Traverser 0 is usually enough for the sweep. Run traverser 1 too if the final choice is close.
        row = profile_once(config, traverser=0, warmup=False)
        sweep_rows.append(row)
        print(
            f"wall={row['wall_s']:.3f}s peak_reserved={row['peak_reserved_GiB']:.2f}GiB "
            f"peak_rows={row['peak_rows']} chunks={row['streamed_edge_chunks']} "
            f"splits={row['streamed_row_splits']} records/s={row['records_per_s']:.0f}"
        )

sweep_df = summarize_profiles(sweep_rows)
sweep_df.style.format(precision=4)

## Root Batch Sweep

Use this only after choosing a row budget/chunk size. This tests how large the root traversal batch should be. Remember that the useful maximum is the eventual `traversals_per_player`: if training uses 4096 traversals/player, root batches above 4096 will not change training.

In [ ]:
ROOT_BATCH_BASE = dict(
    label='root batch sweep: full-first then cap32 hash',
    schedule=(69, 32),
    sample_count=None,
    sample_fraction=None,
    baseline='call',
    sample_mode='hash',
    live_row_budget=131_072,
    action_chunk_size=16_384,
)

ROOT_BATCHES = [1024, 2048, 4096, 8192, 16_384]

root_rows = []
for root_batch in ROOT_BATCHES:
    config = dict(
        ROOT_BATCH_BASE,
        label=f"{ROOT_BATCH_BASE['label']}; root={root_batch}",
        traversal_batch_size=root_batch,
    )
    print('\n--- root batch', root_batch, '---')
    row = profile_once(config, traverser=0, warmup=False)
    root_rows.append(row)
    print(
        f"wall={row['wall_s']:.3f}s peak_reserved={row['peak_reserved_GiB']:.2f}GiB "
        f"records/s={row['records_per_s']:.0f} peak_rows={row['peak_rows']}"
    )

root_df = summarize_profiles(root_rows)
root_df.style.format(precision=4)

## Guardrail Summary

This cell prints a conservative recommendation from the rows you have run. It is not a theorem; use it as a sanity check before starting the overnight notebook.

In [ ]:
frames = []
for name in ('candidate_df', 'sweep_df', 'root_df'):
    if name in globals():
        frame = globals()[name].copy()
        frame['source'] = name
        frames.append(frame)

all_profiles = pd.concat(frames, ignore_index=True) if frames else pd.DataFrame()
if all_profiles.empty:
    print('No profile rows yet.')
else:
    total_gib = torch.cuda.get_device_properties(device).total_memory / 2**30
    memory_limit = 0.75 * total_gib
    usable = all_profiles[
        (all_profiles['peak_reserved_GiB'] < memory_limit)
        & np.isfinite(all_profiles['records_per_s'])
    ].copy()
    usable['memory_headroom_GiB'] = total_gib - usable['peak_reserved_GiB']
    usable = usable.sort_values(['records_per_s', 'peak_reserved_GiB'], ascending=[False, True])
    print(f'Conservative memory limit: {memory_limit:.1f} GiB of {total_gib:.1f} GiB total')
    display(
        usable[
            [
                'source', 'label', 'traversal_batch_size', 'schedule',
                'live_row_budget', 'action_chunk_size', 'wall_s',
                'records_per_s', 'peak_reserved_GiB', 'memory_headroom_GiB',
                'streamed_row_splits', 'streamed_edge_chunks',
                'claim_edge_fraction', 'regret_weight_ESS_fraction',
            ]
        ].head(20).style.format(precision=4)
    )
    if not usable.empty:
        best = usable.iloc[0]
        print('\nFastest safe row:')
        print(best[['label', 'traversal_batch_size', 'schedule', 'live_row_budget', 'action_chunk_size', 'peak_reserved_GiB', 'records_per_s']].to_string())